In [1]:
# ============================================================
# 1. Setup / Import
# ============================================================

import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Linux / nbconvert 実行時にGUIがなくても落ちにくくする
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

import kagglehub

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

Path("results").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)
Path("images").mkdir(exist_ok=True)

print("setup completed")

setup completed


In [2]:
# ============================================================
# 2. Download M5 Forecasting Accuracy Data
# ============================================================

path = kagglehub.competition_download("m5-forecasting-accuracy")

print("Downloaded path:")
print(path)

print("Files:")
print(os.listdir(path))

Downloaded path:
C:\Users\r_miyashita\.cache\kagglehub\competitions\m5-forecasting-accuracy
Files:
['calendar.csv', 'sales_train_evaluation.csv', 'sales_train_validation.csv', 'sample_submission.csv', 'sell_prices.csv']


In [3]:
# ============================================================
# 3. Load CSV files
# ============================================================

calendar = pd.read_csv(os.path.join(path, "calendar.csv"))
sell_prices = pd.read_csv(os.path.join(path, "sell_prices.csv"))
sales = pd.read_csv(os.path.join(path, "sales_train_validation.csv"))

calendar["date"] = pd.to_datetime(calendar["date"])

print("calendar:", calendar.shape)
print("sell_prices:", sell_prices.shape)
print("sales:", sales.shape)

display(calendar.head())
display(sell_prices.head())
display(sales.head())

calendar: (1969, 14)
sell_prices: (6841121, 4)
sales: (30490, 1919)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,d_11,d_12,d_13,d_14,d_15,d_16,d_17,d_18,d_19,d_20,d_21,d_22,d_23,d_24,d_25,d_26,d_27,d_28,d_29,d_30,d_31,d_32,d_33,d_34,d_35,d_36,d_37,d_38,d_39,d_40,d_41,d_42,d_43,d_44,...,d_1864,d_1865,d_1866,d_1867,d_1868,d_1869,d_1870,d_1871,d_1872,d_1873,d_1874,d_1875,d_1876,d_1877,d_1878,d_1879,d_1880,d_1881,d_1882,d_1883,d_1884,d_1885,d_1886,d_1887,d_1888,d_1889,d_1890,d_1891,d_1892,d_1893,d_1894,d_1895,d_1896,d_1897,d_1898,d_1899,d_1900,d_1901,d_1902,d_1903,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,1,1,3,0,0,0,1,1,1,3,1,3,1,2,2,0,1,1,1,1,0,0,0,0,0,1,0,4,2,3,0,1,2,0,0,0,1,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,1,2,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,2,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,2,0,1,0,...,5,3,1,0,0,0,1,2,3,0,1,3,4,2,1,4,1,3,5,0,6,6,0,0,0,0,3,1,2,1,3,1,0,2,5,4,2,0,3,0,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,2,1,1,0,3,1,1,2,1,1,0,3,2,2,2,3,1,0,0,0,0,1,0,4,4,0,1,4,0,1,0,1,0,1,1,2,0,1,1,2,1,1,0,1,1,2,2,2,4


In [4]:
# ============================================================
# 4. Filter FOODS category
# ============================================================

foods_sales = sales[sales["cat_id"] == "FOODS"].copy()

print("foods_sales:", foods_sales.shape)

display(
    foods_sales[
        ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    ].head()
)

display(foods_sales["dept_id"].value_counts())

foods_sales: (14370, 1919)


,id,item_id,dept_id,cat_id,store_id,state_id
1612,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
1613,FOODS_1_002_CA_1_validation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA
1614,FOODS_1_003_CA_1_validation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA
1615,FOODS_1_004_CA_1_validation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA
1616,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA


dept_id
FOODS_3    8230
FOODS_2    3980
FOODS_1    2160
Name: count, dtype: int64

In [5]:
# ============================================================
# 5. Create sales volume groups: top / middle / bottom
# ============================================================

day_cols = [col for col in foods_sales.columns if col.startswith("d_")]

item_total_sales = foods_sales.groupby(
    ["item_id", "dept_id", "cat_id"]
)[day_cols].sum()

item_total_sales["total_sales"] = item_total_sales.sum(axis=1)

item_total_sales = (
    item_total_sales
    .reset_index()
    .sort_values("total_sales", ascending=False)
    .reset_index(drop=True)
)

display(item_total_sales.head())
display(item_total_sales.tail())

top_items = item_total_sales.head(30)["item_id"].tolist()

middle_start = len(item_total_sales) // 2 - 15
middle_items = item_total_sales.iloc[middle_start:middle_start + 30]["item_id"].tolist()

nonzero_items = item_total_sales[item_total_sales["total_sales"] > 0].copy()
bottom_items = nonzero_items.tail(30)["item_id"].tolist()

print("top:", len(top_items))
print("middle:", len(middle_items))
print("bottom:", len(bottom_items))

,item_id,dept_id,cat_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,d_11,d_12,d_13,d_14,d_15,d_16,d_17,d_18,d_19,d_20,d_21,d_22,d_23,d_24,d_25,d_26,d_27,d_28,d_29,d_30,d_31,d_32,d_33,d_34,d_35,d_36,d_37,d_38,d_39,d_40,d_41,d_42,d_43,d_44,d_45,d_46,d_47,...,d_1865,d_1866,d_1867,d_1868,d_1869,d_1870,d_1871,d_1872,d_1873,d_1874,d_1875,d_1876,d_1877,d_1878,d_1879,d_1880,d_1881,d_1882,d_1883,d_1884,d_1885,d_1886,d_1887,d_1888,d_1889,d_1890,d_1891,d_1892,d_1893,d_1894,d_1895,d_1896,d_1897,d_1898,d_1899,d_1900,d_1901,d_1902,d_1903,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913,total_sales
0,FOODS_3_090,FOODS_3,FOODS,1046,1036,673,642,531,877,1117,1311,1306,517,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,506,414,445,488,605,841,632,521,362,502,423,519,727,604,342,384,399,430,773,1002,1051,321,317,425,481,630,717,657,434,397,428,423,655,827,710,456,395,394,426,672,832,789,486,424,425,422,629,793,592,1002529
1,FOODS_3_586,FOODS_3,FOODS,516,479,328,376,319,430,405,624,537,432,386,391,369,529,713,713,452,462,385,399,489,608,564,378,434,397,381,488,568,568,357,448,416,454,508,618,617,410,431,416,440,467,625,556,413,450,395,...,383,354,367,402,392,508,481,403,363,354,393,367,419,434,313,335,361,391,504,635,579,339,290,300,332,405,485,506,362,411,366,427,409,511,502,381,306,343,369,396,481,530,391,434,373,365,381,477,449,920242
2,FOODS_3_252,FOODS_3,FOODS,289,273,157,172,132,218,227,363,255,204,179,183,216,256,449,429,248,260,208,208,258,367,310,265,212,199,189,256,332,336,256,262,240,230,257,340,315,193,223,215,205,279,370,302,232,211,257,...,215,193,184,238,298,359,334,270,267,236,302,310,357,306,195,221,212,232,417,466,394,216,202,176,245,288,413,374,251,271,276,283,297,368,309,259,206,227,195,271,398,377,259,279,224,249,255,440,287,565299
3,FOODS_3_555,FOODS_3,FOODS,321,315,214,204,159,265,240,425,355,224,253,190,235,325,438,388,211,235,249,268,347,381,309,246,241,221,243,292,344,382,193,289,263,274,322,401,383,221,250,292,278,333,424,358,261,297,295,...,213,180,169,201,211,310,297,234,198,205,176,246,266,207,165,191,186,202,310,351,347,205,176,167,202,253,345,312,211,201,215,232,240,324,294,231,184,159,190,237,315,288,211,230,209,215,247,303,238,491287
4,FOODS_3_714,FOODS_3,FOODS,238,215,141,144,96,139,169,277,219,166,143,136,163,214,346,311,198,186,144,151,197,266,229,183,172,160,147,218,230,237,160,170,158,189,235,296,254,165,183,143,163,212,271,239,182,169,181,...,178,157,185,178,208,261,186,186,164,163,172,196,215,209,132,168,161,150,254,282,207,122,176,147,146,231,257,250,188,210,180,179,203,234,201,156,141,165,186,250,286,285,183,153,136,169,205,258,205,396172


,item_id,dept_id,cat_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,d_11,d_12,d_13,d_14,d_15,d_16,d_17,d_18,d_19,d_20,d_21,d_22,d_23,d_24,d_25,d_26,d_27,d_28,d_29,d_30,d_31,d_32,d_33,d_34,d_35,d_36,d_37,d_38,d_39,d_40,d_41,d_42,d_43,d_44,d_45,d_46,d_47,...,d_1865,d_1866,d_1867,d_1868,d_1869,d_1870,d_1871,d_1872,d_1873,d_1874,d_1875,d_1876,d_1877,d_1878,d_1879,d_1880,d_1881,d_1882,d_1883,d_1884,d_1885,d_1886,d_1887,d_1888,d_1889,d_1890,d_1891,d_1892,d_1893,d_1894,d_1895,d_1896,d_1897,d_1898,d_1899,d_1900,d_1901,d_1902,d_1903,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913,total_sales
1432,FOODS_2_071,FOODS_2,FOODS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,8,2,1,2,6,1,1,11,5,6,4,2,10,1,2,2,3,2,6,7,1,6,2,2,2,1,2,4,4,4,0,5,2,2,5,0,3,3,7,3,1,7,3,3,4,4,3,3,6,1123
1433,FOODS_3_171,FOODS_3,FOODS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,2,2,0,0,1,1,1,0,1,1,0,1,1,0,2,0,1,0,1,1,0,1,2,1,1,4,3,1,0,0,0,1,1,1,0,0,0,2,0,2,0,1121
1434,FOODS_2_073,FOODS_2,FOODS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,3,0,0,3,5,0,0,5,0,0,3,1,3,2,0,2,1,1,2,1,2,0,3,2,2,4,1,0,3,0,5,2,5,6,3,1,3,2,0,0,4,1,2,1,2,0,3,0,1094
1435,FOODS_3_472,FOODS_3,FOODS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,3,2,2,2,2,0,5,1,0,1,1,2,7,3,0,4,5,5,0,7,4,2,1,0,2,3,7,3,0,2,7,4,2,5,5,5,6,2,1,5,6,5,4,3,5,1,4,6,8,1043
1436,FOODS_3_220,FOODS_3,FOODS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,3,5,5,2,1,1,6,7,2,1,2,1,4,3,2,2,1,5,1,5,5,1,1,2,2,3,6,4,2,0,2,6,3,4,2,3,0,1,0,1,4,2,1,4,2,1,3,1,1,823


top: 30
middle: 30
bottom: 30


In [6]:
# ============================================================
# 6. Assign sales_group
# ============================================================

def assign_sales_group(item_id):
    if item_id in top_items:
        return "top"
    elif item_id in middle_items:
        return "middle"
    elif item_id in bottom_items:
        return "bottom"
    else:
        return "other"


foods_sales["sales_group"] = foods_sales["item_id"].apply(assign_sales_group)

target_sales = foods_sales[foods_sales["sales_group"] != "other"].copy()

print("target_sales:", target_sales.shape)

display(target_sales["sales_group"].value_counts())
display(target_sales["dept_id"].value_counts())

target_sales: (900, 1920)


sales_group
middle    300
bottom    300
top       300
Name: count, dtype: int64

dept_id
FOODS_3    650
FOODS_2    190
FOODS_1     60
Name: count, dtype: int64

In [7]:
# ============================================================
# 7. Reshape sales data from wide to long
# ============================================================

id_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "sales_group"
]

long_df = target_sales.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

print("long_df:", long_df.shape)
display(long_df.head())

long_df: (1721700, 9)


,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2
1,FOODS_1_079_CA_1_validation,FOODS_1_079,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0
2,FOODS_1_152_CA_1_validation,FOODS_1_152,FOODS_1,FOODS,CA_1,CA,middle,d_1,2
3,FOODS_1_160_CA_1_validation,FOODS_1_160,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0
4,FOODS_1_168_CA_1_validation,FOODS_1_168,FOODS_1,FOODS,CA_1,CA,middle,d_1,0


In [8]:
# ============================================================
# 8. Merge calendar data
# ============================================================

calendar_cols = [
    "d",
    "date",
    "wm_yr_wk",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

model_df = long_df.merge(
    calendar[calendar_cols],
    on="d",
    how="left"
)

model_df["date"] = pd.to_datetime(model_df["date"])

print(model_df.shape)
display(model_df.head())

(1721700, 21)


,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,FOODS_1_079_CA_1_validation,FOODS_1_079,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,FOODS_1_152_CA_1_validation,FOODS_1_152,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,FOODS_1_160_CA_1_validation,FOODS_1_160,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
4,FOODS_1_168_CA_1_validation,FOODS_1_168,FOODS_1,FOODS,CA_1,CA,middle,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0


In [9]:
# ============================================================
# 9. Merge sell_prices data
# ============================================================

model_df = model_df.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

print(model_df.shape)
print("sell_price missing rate:", model_df["sell_price"].isna().mean())

display(model_df.head())

(1721700, 22)
sell_price missing rate: 0.28053261311494454


,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18
1,FOODS_1_079_CA_1_validation,FOODS_1_079,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
2,FOODS_1_152_CA_1_validation,FOODS_1_152,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,6.98
3,FOODS_1_160_CA_1_validation,FOODS_1_160,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
4,FOODS_1_168_CA_1_validation,FOODS_1_168,FOODS_1,FOODS,CA_1,CA,middle,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.00


In [10]:
# ============================================================
# 10. Date features
# ============================================================

model_df["day"] = model_df["date"].dt.day
model_df["dayofweek"] = model_df["date"].dt.dayofweek
model_df["weekofyear"] = model_df["date"].dt.isocalendar().week.astype(int)
model_df["quarter"] = model_df["date"].dt.quarter

model_df["is_weekend"] = model_df["dayofweek"].isin([5, 6]).astype(int)
model_df["is_month_start"] = model_df["date"].dt.is_month_start.astype(int)
model_df["is_month_end"] = model_df["date"].dt.is_month_end.astype(int)

model_df["week_of_month"] = ((model_df["day"] - 1) // 7 + 1).astype(int)

model_df["is_week_1"] = (model_df["week_of_month"] == 1).astype(int)
model_df["is_week_2"] = (model_df["week_of_month"] == 2).astype(int)
model_df["is_week_3"] = (model_df["week_of_month"] == 3).astype(int)
model_df["is_week_4"] = (model_df["week_of_month"] == 4).astype(int)
model_df["is_week_5"] = (model_df["week_of_month"] >= 5).astype(int)

display(model_df.head())

,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day,dayofweek,weekofyear,quarter,is_weekend,is_month_start,is_month_end,week_of_month,is_week_1,is_week_2,is_week_3,is_week_4,is_week_5
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,29,5,4,1,1,0,0,5,0,0,0,0,1
1,FOODS_1_079_CA_1_validation,FOODS_1_079,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN,29,5,4,1,1,0,0,5,0,0,0,0,1
2,FOODS_1_152_CA_1_validation,FOODS_1_152,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,6.98,29,5,4,1,1,0,0,5,0,0,0,0,1
3,FOODS_1_160_CA_1_validation,FOODS_1_160,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN,29,5,4,1,1,0,0,5,0,0,0,0,1
4,FOODS_1_168_CA_1_validation,FOODS_1_168,FOODS_1,FOODS,CA_1,CA,middle,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.00,29,5,4,1,1,0,0,5,0,0,0,0,1


In [11]:
# ============================================================
# 11. Event features
# ============================================================

model_df["has_event_1"] = model_df["event_name_1"].notna().astype(int)
model_df["has_event_2"] = model_df["event_name_2"].notna().astype(int)

model_df["has_event"] = (
    (model_df["has_event_1"] == 1) |
    (model_df["has_event_2"] == 1)
).astype(int)

event_type_1_dummies = pd.get_dummies(
    model_df["event_type_1"].fillna("NoEvent"),
    prefix="event_type_1",
    dtype=int
)

event_type_2_dummies = pd.get_dummies(
    model_df["event_type_2"].fillna("NoEvent"),
    prefix="event_type_2",
    dtype=int
)

model_df = pd.concat(
    [model_df, event_type_1_dummies, event_type_2_dummies],
    axis=1
)

display(model_df.head())

,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day,dayofweek,weekofyear,quarter,is_weekend,is_month_start,is_month_end,week_of_month,is_week_1,is_week_2,is_week_3,is_week_4,is_week_5,has_event_1,has_event_2,has_event,event_type_1_Cultural,event_type_1_National,event_type_1_NoEvent,event_type_1_Religious,event_type_1_Sporting,event_type_2_Cultural,event_type_2_NoEvent,event_type_2_Religious
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
1,FOODS_1_079_CA_1_validation,FOODS_1_079,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
2,FOODS_1_152_CA_1_validation,FOODS_1_152,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,6.98,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
3,FOODS_1_160_CA_1_validation,FOODS_1_160,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
4,FOODS_1_168_CA_1_validation,FOODS_1_168,FOODS_1,FOODS,CA_1,CA,middle,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.00,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0


In [12]:
# ============================================================
# 12. SNAP feature
# ============================================================

def get_snap(row):
    if row["state_id"] == "CA":
        return row["snap_CA"]
    elif row["state_id"] == "TX":
        return row["snap_TX"]
    elif row["state_id"] == "WI":
        return row["snap_WI"]
    else:
        return 0


model_df["snap"] = model_df.apply(get_snap, axis=1).astype(int)

display(model_df[["state_id", "snap_CA", "snap_TX", "snap_WI", "snap"]].head())

,state_id,snap_CA,snap_TX,snap_WI,snap
0,CA,0,0,0,0
1,CA,0,0,0,0
2,CA,0,0,0,0
3,CA,0,0,0,0
4,CA,0,0,0,0


In [13]:
# ============================================================
# 13. One-hot encoding
# ============================================================

dept_dummies = pd.get_dummies(model_df["dept_id"], prefix="dept", dtype=int)
store_dummies = pd.get_dummies(model_df["store_id"], prefix="store", dtype=int)
state_dummies = pd.get_dummies(model_df["state_id"], prefix="state", dtype=int)
dow_dummies = pd.get_dummies(model_df["dayofweek"], prefix="dow", dtype=int)

model_df = pd.concat(
    [model_df, dept_dummies, store_dummies, state_dummies, dow_dummies],
    axis=1
)

print("model_df:", model_df.shape)
display(model_df.head())

model_df: (1721700, 70)


,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day,dayofweek,weekofyear,quarter,is_weekend,is_month_start,is_month_end,week_of_month,is_week_1,is_week_2,is_week_3,is_week_4,is_week_5,has_event_1,has_event_2,has_event,event_type_1_Cultural,event_type_1_National,event_type_1_NoEvent,event_type_1_Religious,event_type_1_Sporting,event_type_2_Cultural,event_type_2_NoEvent,event_type_2_Religious,snap,dept_FOODS_1,dept_FOODS_2,dept_FOODS_3,store_CA_1,store_CA_2,store_CA_3,store_CA_4,store_TX_1,store_TX_2,store_TX_3,store_WI_1,store_WI_2,store_WI_3,state_CA,state_TX,state_WI,dow_0,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
1,FOODS_1_079_CA_1_validation,FOODS_1_079,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
2,FOODS_1_152_CA_1_validation,FOODS_1_152,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,6.98,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
3,FOODS_1_160_CA_1_validation,FOODS_1_160,FOODS_1,FOODS,CA_1,CA,bottom,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
4,FOODS_1_168_CA_1_validation,FOODS_1_168,FOODS_1,FOODS,CA_1,CA,middle,d_1,0,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.00,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0


In [14]:
# ============================================================
# 14. Lag / rolling / zero-rate features
# ============================================================

model_df = model_df.sort_values(["id", "date"]).reset_index(drop=True)

lag_days = [7, 14, 21, 28]

for lag in lag_days:
    model_df[f"lag_{lag}"] = (
        model_df.groupby("id")["sales"].shift(lag)
    )

rolling_windows = [7, 14, 28]

for window in rolling_windows:
    model_df[f"rolling_mean_{window}"] = (
        model_df.groupby("id")["sales"]
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )

    model_df[f"rolling_std_{window}"] = (
        model_df.groupby("id")["sales"]
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

    model_df[f"zero_rate_{window}"] = (
        model_df.groupby("id")["sales"]
        .transform(lambda x: x.shift(1).rolling(window).apply(lambda s: (s == 0).mean()))
    )

display(
    model_df[
        [
            "id",
            "date",
            "sales",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_28",
            "zero_rate_28"
        ]
    ].head(40)
)

,id,date,sales,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,zero_rate_28
0,FOODS_1_041_CA_1_validation,2011-01-29,2,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_041_CA_1_validation,2011-01-30,0,NaN,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_041_CA_1_validation,2011-01-31,0,NaN,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_041_CA_1_validation,2011-02-01,1,NaN,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_041_CA_1_validation,2011-02-02,0,NaN,NaN,NaN,NaN,NaN,NaN
5,FOODS_1_041_CA_1_validation,2011-02-03,1,NaN,NaN,NaN,NaN,NaN,NaN
6,FOODS_1_041_CA_1_validation,2011-02-04,3,NaN,NaN,NaN,NaN,NaN,NaN
7,FOODS_1_041_CA_1_validation,2011-02-05,1,2.0,NaN,NaN,1.000000,NaN,NaN
8,FOODS_1_041_CA_1_validation,2011-02-06,7,0.0,NaN,NaN,0.857143,NaN,NaN
9,FOODS_1_041_CA_1_validation,2011-02-07,2,0.0,NaN,NaN,1.857143,NaN,NaN


In [15]:
# ============================================================
# 15. Price features
# ============================================================

model_df["sell_price"] = (
    model_df.groupby("id")["sell_price"]
    .ffill()
)

model_df["sell_price"] = model_df["sell_price"].fillna(model_df["sell_price"].median())

model_df["price_lag_1"] = (
    model_df.groupby("id")["sell_price"].shift(1)
)

model_df["price_diff_1"] = (
    model_df["sell_price"] - model_df["price_lag_1"]
)

model_df["price_pct_change_1"] = (
    model_df["price_diff_1"] / (model_df["price_lag_1"] + 1e-6)
)

model_df["is_price_down"] = (model_df["price_diff_1"] < 0).astype(int)
model_df["is_price_up"] = (model_df["price_diff_1"] > 0).astype(int)

model_df["price_x_rolling_mean_7"] = (
    model_df["sell_price"] * model_df["rolling_mean_7"]
)

model_df["price_down_x_rolling_mean_7"] = (
    model_df["is_price_down"] * model_df["rolling_mean_7"]
)

display(
    model_df[
        [
            "id",
            "date",
            "sell_price",
            "price_lag_1",
            "price_diff_1",
            "price_pct_change_1",
            "is_price_down",
            "is_price_up"
        ]
    ].head(40)
)

,id,date,sell_price,price_lag_1,price_diff_1,price_pct_change_1,is_price_down,is_price_up
0,FOODS_1_041_CA_1_validation,2011-01-29,2.18,NaN,NaN,NaN,0,0
1,FOODS_1_041_CA_1_validation,2011-01-30,2.18,2.18,0.0,0.0,0,0
2,FOODS_1_041_CA_1_validation,2011-01-31,2.18,2.18,0.0,0.0,0,0
3,FOODS_1_041_CA_1_validation,2011-02-01,2.18,2.18,0.0,0.0,0,0
4,FOODS_1_041_CA_1_validation,2011-02-02,2.18,2.18,0.0,0.0,0,0
5,FOODS_1_041_CA_1_validation,2011-02-03,2.18,2.18,0.0,0.0,0,0
6,FOODS_1_041_CA_1_validation,2011-02-04,2.18,2.18,0.0,0.0,0,0
7,FOODS_1_041_CA_1_validation,2011-02-05,2.18,2.18,0.0,0.0,0,0
8,FOODS_1_041_CA_1_validation,2011-02-06,2.18,2.18,0.0,0.0,0,0
9,FOODS_1_041_CA_1_validation,2011-02-07,2.18,2.18,0.0,0.0,0,0


In [16]:
# ============================================================
# 16. Trend / interaction features
# ============================================================

model_df["rolling_mean_7_minus_28"] = (
    model_df["rolling_mean_7"] - model_df["rolling_mean_28"]
)

model_df["rolling_mean_7_div_28"] = (
    model_df["rolling_mean_7"] / (model_df["rolling_mean_28"] + 1e-6)
)

model_df["lag_7_minus_14"] = model_df["lag_7"] - model_df["lag_14"]
model_df["lag_7_minus_21"] = model_df["lag_7"] - model_df["lag_21"]
model_df["lag_7_minus_28"] = model_df["lag_7"] - model_df["lag_28"]

model_df["weekend_x_rolling_mean_7"] = (
    model_df["is_weekend"] * model_df["rolling_mean_7"]
)

model_df["snap_x_rolling_mean_7"] = (
    model_df["snap"] * model_df["rolling_mean_7"]
)

model_df["event_x_rolling_mean_7"] = (
    model_df["has_event"] * model_df["rolling_mean_7"]
)

model_df["week_of_month_x_rolling_mean_7"] = (
    model_df["week_of_month"] * model_df["rolling_mean_7"]
)

display(model_df.head())

,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day,dayofweek,weekofyear,quarter,is_weekend,is_month_start,is_month_end,week_of_month,is_week_1,is_week_2,is_week_3,is_week_4,is_week_5,has_event_1,has_event_2,has_event,event_type_1_Cultural,event_type_1_National,event_type_1_NoEvent,event_type_1_Religious,event_type_1_Sporting,event_type_2_Cultural,event_type_2_NoEvent,event_type_2_Religious,snap,dept_FOODS_1,dept_FOODS_2,dept_FOODS_3,store_CA_1,store_CA_2,store_CA_3,store_CA_4,store_TX_1,store_TX_2,store_TX_3,store_WI_1,store_WI_2,store_WI_3,state_CA,state_TX,state_WI,dow_0,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_std_7,zero_rate_7,rolling_mean_14,rolling_std_14,zero_rate_14,rolling_mean_28,rolling_std_28,zero_rate_28,price_lag_1,price_diff_1,price_pct_change_1,is_price_down,is_price_up,price_x_rolling_mean_7,price_down_x_rolling_mean_7,rolling_mean_7_minus_28,rolling_mean_7_div_28,lag_7_minus_14,lag_7_minus_21,lag_7_minus_28,weekend_x_rolling_mean_7,snap_x_rolling_mean_7,event_x_rolling_mean_7,week_of_month_x_rolling_mean_7
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_2,0,2011-01-30,11101,2,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,30,6,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.18,0.0,0.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_3,0,2011-01-31,11101,3,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,31,0,5,1,0,0,1,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.18,0.0,0.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_4,1,2011-02-01,11101,4,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.18,1,1,5,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.18,0.0,0.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_5,0,2011-02-02,11101,5,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.18,2,2,5,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.18,0.0,0.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# ============================================================
# 17. Train / test split date
# ============================================================

split_date = pd.to_datetime("2016-03-01")

print("date min:", model_df["date"].min())
print("date max:", model_df["date"].max())
print("split_date:", split_date)

date min: 2011-01-29 00:00:00
date max: 2016-04-24 00:00:00
split_date: 2016-03-01 00:00:00


In [18]:
# ============================================================
# 18. Day-of-week effect features
# Only training period is used to avoid leakage
# ============================================================

train_for_dow = model_df[model_df["date"] < split_date].copy()

group_dow_mean = train_for_dow.groupby(
    ["sales_group", "dayofweek"]
).agg(
    group_dow_sales_mean=("sales", "mean")
).reset_index()

group_mean = train_for_dow.groupby("sales_group").agg(
    group_sales_mean=("sales", "mean")
).reset_index()

group_dow_feature = group_dow_mean.merge(
    group_mean,
    on="sales_group",
    how="left"
)

group_dow_feature["group_dow_sales_ratio"] = (
    group_dow_feature["group_dow_sales_mean"] /
    group_dow_feature["group_sales_mean"]
)

display(group_dow_feature)

model_df = model_df.merge(
    group_dow_feature[
        [
            "sales_group",
            "dayofweek",
            "group_dow_sales_mean",
            "group_dow_sales_ratio"
        ]
    ],
    on=["sales_group", "dayofweek"],
    how="left"
)

model_df["dow_ratio_x_rolling_mean_7"] = (
    model_df["group_dow_sales_ratio"] * model_df["rolling_mean_7"]
)

display(
    model_df[
        [
            "sales_group",
            "dayofweek",
            "group_dow_sales_mean",
            "group_dow_sales_ratio",
            "rolling_mean_7",
            "dow_ratio_x_rolling_mean_7"
        ]
    ].head()
)

,sales_group,dayofweek,group_dow_sales_mean,group_sales_mean,group_dow_sales_ratio
0,bottom,0,0.071278,0.074386,0.958215
1,bottom,1,0.065321,0.074386,0.878127
2,bottom,2,0.065862,0.074386,0.885398
3,bottom,3,0.065208,0.074386,0.876605
4,bottom,4,0.071572,0.074386,0.962169
5,bottom,5,0.088484,0.074386,1.189514
6,bottom,6,0.092870,0.074386,1.248476
7,middle,0,0.752957,0.773577,0.973345
8,middle,1,0.682516,0.773577,0.882285
9,middle,2,0.674792,0.773577,0.872301


,sales_group,dayofweek,group_dow_sales_mean,group_dow_sales_ratio,rolling_mean_7,dow_ratio_x_rolling_mean_7
0,middle,5,0.923797,1.194188,NaN,NaN
1,middle,6,0.952657,1.231495,NaN,NaN
2,middle,0,0.752957,0.973345,NaN,NaN
3,middle,1,0.682516,0.882285,NaN,NaN
4,middle,2,0.674792,0.872301,NaN,NaN


In [19]:
# ============================================================
# 19. Business summary tables
# ============================================================

dept_sales = model_df.groupby(["sales_group", "dept_id"]).agg(
    sales_mean=("sales", "mean"),
    sales_median=("sales", "median"),
    sales_count=("sales", "count")
).reset_index()

week_of_month_sales = model_df.groupby(["sales_group", "week_of_month"]).agg(
    sales_mean=("sales", "mean"),
    sales_median=("sales", "median"),
    sales_count=("sales", "count")
).reset_index()

event_sales = model_df.groupby(["sales_group", "has_event"]).agg(
    sales_mean=("sales", "mean"),
    sales_median=("sales", "median"),
    sales_count=("sales", "count")
).reset_index()

snap_sales = model_df.groupby(["sales_group", "snap"]).agg(
    sales_mean=("sales", "mean"),
    sales_median=("sales", "median"),
    sales_count=("sales", "count")
).reset_index()

price_down_sales = model_df.groupby(["sales_group", "is_price_down"]).agg(
    sales_mean=("sales", "mean"),
    sales_median=("sales", "median"),
    sales_count=("sales", "count")
).reset_index()

display(dept_sales)
display(week_of_month_sales)
display(event_sales)
display(snap_sales)
display(price_down_sales)

,sales_group,dept_id,sales_mean,sales_median,sales_count
0,bottom,FOODS_1,0.096158,0.0,38260
1,bottom,FOODS_2,0.077073,0.0,191300
2,bottom,FOODS_3,0.083194,0.0,344340
3,middle,FOODS_1,0.770936,0.0,57390
4,middle,FOODS_2,0.785226,0.0,153040
5,middle,FOODS_3,0.786106,0.0,363470
6,top,FOODS_1,12.263147,9.0,19130
7,top,FOODS_2,13.440617,9.0,19130
8,top,FOODS_3,17.542807,11.0,535640


,sales_group,week_of_month,sales_mean,sales_median,sales_count
0,bottom,1,0.085185,0.0,132300
1,bottom,2,0.086772,0.0,132300
2,bottom,3,0.082676,0.0,132300
3,bottom,4,0.076064,0.0,131100
4,bottom,5,0.074292,0.0,45900
5,middle,1,0.836856,0.0,132300
6,middle,2,0.850899,0.0,132300
7,middle,3,0.777029,0.0,132300
8,middle,4,0.700320,0.0,131100
9,middle,5,0.702353,0.0,45900


,sales_group,has_event,sales_mean,sales_median,sales_count
0,bottom,0,0.082331,0.0,527700
1,bottom,1,0.078442,0.0,46200
2,middle,0,0.787654,0.0,527700
3,middle,1,0.746667,0.0,46200
4,top,0,17.252295,11.0,527700
5,top,1,16.976320,10.0,46200


,sales_group,snap,sales_mean,sales_median,sales_count
0,bottom,0,0.078613,0.0,384900
1,bottom,1,0.088952,0.0,189000
2,middle,0,0.737685,0.0,384900
3,middle,1,0.879397,0.0,189000
4,top,0,16.625944,11.0,384900
5,top,1,18.460402,12.0,189000


,sales_group,is_price_down,sales_mean,sales_median,sales_count
0,bottom,0,0.081957,0.0,573712
1,bottom,1,0.265957,0.0,188
2,middle,0,0.784461,0.0,573382
3,middle,1,0.666023,0.0,518
4,top,0,17.239266,11.0,573237
5,top,1,9.286576,0.0,663


In [20]:
# ============================================================
# 20. Feature lists
# ============================================================

date_features = [
    "wday",
    "month",
    "year",
    "day",
    "dayofweek",
    "weekofyear",
    "quarter",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "week_of_month",
    "is_week_1",
    "is_week_2",
    "is_week_3",
    "is_week_4",
    "is_week_5"
]

lag_features = [
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28"
]

rolling_features = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28",
    "zero_rate_7",
    "zero_rate_14",
    "zero_rate_28"
]

trend_features = [
    "rolling_mean_7_minus_28",
    "rolling_mean_7_div_28",
    "lag_7_minus_14",
    "lag_7_minus_21",
    "lag_7_minus_28"
]

dow_effect_features = [
    "group_dow_sales_mean",
    "group_dow_sales_ratio",
    "dow_ratio_x_rolling_mean_7",
    "weekend_x_rolling_mean_7",
    "week_of_month_x_rolling_mean_7"
]

price_features = [
    "sell_price",
    "price_lag_1",
    "price_diff_1",
    "price_pct_change_1",
    "is_price_down",
    "is_price_up",
    "price_x_rolling_mean_7",
    "price_down_x_rolling_mean_7"
]

event_snap_features = [
    "has_event_1",
    "has_event_2",
    "has_event",
    "snap",
    "snap_x_rolling_mean_7",
    "event_x_rolling_mean_7"
]

onehot_features = [
    col for col in model_df.columns
    if (
        col.startswith("dept_")
        or col.startswith("store_")
        or col.startswith("state_")
        or col.startswith("dow_")
        or col.startswith("event_type_1_")
        or col.startswith("event_type_2_")
    )
]

# 元の文字列カラムが混ざらないように明示除外
exclude_original_string_cols = {
    "dept_id",
    "store_id",
    "state_id",
    "event_type_1",
    "event_type_2",
    "event_name_1",
    "event_name_2",
    "item_id",
    "cat_id",
    "id",
    "d"
}

onehot_features = [
    col for col in onehot_features
    if col not in exclude_original_string_cols
]

base_features = date_features + lag_features + rolling_features

features_v2 = base_features + trend_features

features_v3 = features_v2 + dow_effect_features

features_v4 = (
    features_v3 +
    price_features +
    event_snap_features +
    onehot_features
)

def clean_feature_list(features, df):
    cleaned = []
    for col in features:
        if col in df.columns and col not in cleaned:
            cleaned.append(col)
    return cleaned


base_features = clean_feature_list(base_features, model_df)
features_v2 = clean_feature_list(features_v2, model_df)
features_v3 = clean_feature_list(features_v3, model_df)
features_v4 = clean_feature_list(features_v4, model_df)

print("base:", len(base_features))
print("v2:", len(features_v2))
print("v3:", len(features_v3))
print("v4:", len(features_v4))

bad_cols = [
    col for col in features_v4
    if model_df[col].dtype == "object" or str(model_df[col].dtype) == "string"
]

print("bad object columns in features_v4:", bad_cols)

base: 29
v2: 34
v3: 39
v4: 84
bad object columns in features_v4: []


In [21]:
# ============================================================
# 21. Prepare ML data
# ============================================================

model_df_ml = model_df.loc[:, ~model_df.columns.duplicated()].copy()

model_df_ml["date"] = pd.to_datetime(model_df_ml["date"])

model_df_ml = model_df_ml.replace([np.inf, -np.inf], np.nan)

# feature欠損は0埋め
model_df_ml[features_v4] = model_df_ml[features_v4].fillna(0)

# 必須列のみdropna
model_df_ml = model_df_ml.dropna(
    subset=["date", "sales", "sales_group"]
).copy()

# 念のため、数値以外の特徴量を除外
def keep_numeric_features(features, df):
    numeric_features = []
    dropped = []

    for col in features:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_features.append(col)
        else:
            dropped.append(col)

    if dropped:
        print("Dropped non-numeric features:", dropped)

    return numeric_features


base_features = keep_numeric_features(base_features, model_df_ml)
features_v2 = keep_numeric_features(features_v2, model_df_ml)
features_v3 = keep_numeric_features(features_v3, model_df_ml)
features_v4 = keep_numeric_features(features_v4, model_df_ml)

print("model_df_ml shape:", model_df_ml.shape)

print("date range:")
print(model_df_ml["date"].min(), model_df_ml["date"].max())

print("group counts:")
display(model_df_ml["sales_group"].value_counts())

print("test counts by group:")
display(
    model_df_ml[model_df_ml["date"] >= split_date]
    .groupby("sales_group")
    .size()
)

print("feature dtype counts:")
display(model_df_ml[features_v4].dtypes.value_counts())

display(model_df_ml.head())

model_df_ml shape: (1721700, 102)
date range:
2011-01-29 00:00:00 2016-04-24 00:00:00
group counts:


sales_group
middle    573900
bottom    573900
top       573900
Name: count, dtype: int64

test counts by group:


sales_group
bottom    16500
middle    16500
top       16500
dtype: int64

feature dtype counts:


int64      50
float64    31
int32       3
Name: count, dtype: int64

,id,item_id,dept_id,cat_id,store_id,state_id,sales_group,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,day,dayofweek,weekofyear,quarter,is_weekend,is_month_start,is_month_end,week_of_month,is_week_1,is_week_2,is_week_3,is_week_4,is_week_5,has_event_1,has_event_2,has_event,event_type_1_Cultural,event_type_1_National,event_type_1_NoEvent,event_type_1_Religious,event_type_1_Sporting,event_type_2_Cultural,event_type_2_NoEvent,event_type_2_Religious,snap,dept_FOODS_1,dept_FOODS_2,dept_FOODS_3,...,store_CA_3,store_CA_4,store_TX_1,store_TX_2,store_TX_3,store_WI_1,store_WI_2,store_WI_3,state_CA,state_TX,state_WI,dow_0,dow_1,dow_2,dow_3,dow_4,dow_5,dow_6,lag_7,lag_14,lag_21,lag_28,rolling_mean_7,rolling_std_7,zero_rate_7,rolling_mean_14,rolling_std_14,zero_rate_14,rolling_mean_28,rolling_std_28,zero_rate_28,price_lag_1,price_diff_1,price_pct_change_1,is_price_down,is_price_up,price_x_rolling_mean_7,price_down_x_rolling_mean_7,rolling_mean_7_minus_28,rolling_mean_7_div_28,lag_7_minus_14,lag_7_minus_21,lag_7_minus_28,weekend_x_rolling_mean_7,snap_x_rolling_mean_7,event_x_rolling_mean_7,week_of_month_x_rolling_mean_7,group_dow_sales_mean,group_dow_sales_ratio,dow_ratio_x_rolling_mean_7
0,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_1,2,2011-01-29,11101,1,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,29,5,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.923797,1.194188,0.0
1,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_2,0,2011-01-30,11101,2,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,30,6,4,1,1,0,0,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.18,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.952657,1.231495,0.0
2,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_3,0,2011-01-31,11101,3,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.18,31,0,5,1,0,0,1,5,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.18,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.752957,0.973345,0.0
3,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_4,1,2011-02-01,11101,4,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.18,1,1,5,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.18,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.682516,0.882285,0.0
4,FOODS_1_041_CA_1_validation,FOODS_1_041,FOODS_1,FOODS,CA_1,CA,middle,d_5,0,2011-02-02,11101,5,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.18,2,2,5,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.18,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.674792,0.872301,0.0


In [22]:
# ============================================================
# 22. Baseline evaluation
# ============================================================

def evaluate_baselines(df, group_name, split_date):
    group_df = df[df["sales_group"] == group_name].copy()
    test_df = group_df[group_df["date"] >= split_date].copy()

    if len(test_df) == 0:
        print(f"{group_name}: test data is empty")
        return pd.DataFrame()

    y_test = test_df["sales"]

    baselines = {
        "naive_lag_7": test_df["lag_7"],
        "rolling_mean_7": test_df["rolling_mean_7"],
        "rolling_mean_28": test_df["rolling_mean_28"]
    }

    rows = []

    for model_name, preds in baselines.items():
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        sales_mean = y_test.mean()

        rows.append({
            "group": group_name,
            "model": model_name,
            "mae": mae,
            "rmse": rmse,
            "test_sales_mean": sales_mean,
            "mae_ratio": mae / sales_mean if sales_mean != 0 else np.nan,
            "rmse_ratio": rmse / sales_mean if sales_mean != 0 else np.nan
        })

    return pd.DataFrame(rows)


baseline_results = pd.concat([
    evaluate_baselines(model_df_ml, "top", split_date),
    evaluate_baselines(model_df_ml, "middle", split_date),
    evaluate_baselines(model_df_ml, "bottom", split_date)
], ignore_index=True)

baseline_results = baseline_results.sort_values(
    ["group", "mae_ratio"]
).reset_index(drop=True)

display(baseline_results)

,group,model,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,rolling_mean_7,0.426502,0.760954,0.339818,1.255089,2.239298
1,bottom,rolling_mean_28,0.431543,0.748319,0.339818,1.269924,2.202117
2,bottom,naive_lag_7,0.476727,1.016053,0.339818,1.402889,2.989990
3,middle,rolling_mean_28,1.000628,1.461783,1.148424,0.871305,1.272859
4,middle,rolling_mean_7,1.014978,1.493766,1.148424,0.883801,1.300709
5,middle,naive_lag_7,1.216788,1.931635,1.148424,1.059528,1.681987
6,top,rolling_mean_7,5.956970,9.916855,16.138667,0.369112,0.614478
7,top,rolling_mean_28,6.393429,10.301241,16.138667,0.396156,0.638296
8,top,naive_lag_7,7.007939,11.746786,16.138667,0.434233,0.727866


In [23]:
# ============================================================
# 23. Training function
# ============================================================

def train_ml_model(df, group_name, features, model_name, split_date):
    group_df = df[df["sales_group"] == group_name].copy()

    train_df = group_df[group_df["date"] < split_date].copy()
    test_df = group_df[group_df["date"] >= split_date].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        print(f"{group_name} {model_name}: train or test is empty")
        return None

    X_train = train_df[features]
    y_train = train_df["sales"]

    X_test = test_df[features]
    y_test = test_df["sales"]

    if model_name == "RandomForest":
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "XGBoost":
        model = XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1
        )

    else:
        raise ValueError("model_name must be RandomForest or XGBoost")

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    sales_mean = y_test.mean()

    importance_df = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_,
        "group": group_name,
        "model": model_name
    }).sort_values(
        "importance",
        ascending=False
    ).reset_index(drop=True)

    return {
        "group": group_name,
        "model": model_name,
        "mae": mae,
        "rmse": rmse,
        "test_sales_mean": sales_mean,
        "mae_ratio": mae / sales_mean if sales_mean != 0 else np.nan,
        "rmse_ratio": rmse / sales_mean if sales_mean != 0 else np.nan,
        "trained_model": model,
        "importance": importance_df,
        "test_df": test_df,
        "preds": preds
    }

In [24]:
# ============================================================
# 24. Ablation Study
# ============================================================

feature_sets = {
    "base_lag_rolling": base_features,
    "v2_trend": features_v2,
    "v3_dow_effect": features_v3,
    "v4_price_event_dept": features_v4
}

all_results = []

for group_name in ["top", "middle", "bottom"]:
    for feature_set_name, feature_list in feature_sets.items():
        print(f"Training: {group_name} / {feature_set_name}")

        result = train_ml_model(
            df=model_df_ml,
            group_name=group_name,
            features=feature_list,
            model_name="XGBoost",
            split_date=split_date
        )

        if result is not None:
            result["feature_set"] = feature_set_name
            all_results.append(result)

feature_compare_summary = pd.DataFrame([
    {
        "group": r["group"],
        "feature_set": r["feature_set"],
        "model": r["model"],
        "mae": r["mae"],
        "rmse": r["rmse"],
        "test_sales_mean": r["test_sales_mean"],
        "mae_ratio": r["mae_ratio"],
        "rmse_ratio": r["rmse_ratio"]
    }
    for r in all_results
])

feature_compare_summary = feature_compare_summary.sort_values(
    ["group", "mae_ratio"]
).reset_index(drop=True)

display(feature_compare_summary)

Training: top / base_lag_rolling
Training: top / v2_trend
Training: top / v3_dow_effect
Training: top / v4_price_event_dept
Training: middle / base_lag_rolling
Training: middle / v2_trend
Training: middle / v3_dow_effect
Training: middle / v4_price_event_dept
Training: bottom / base_lag_rolling
Training: bottom / v2_trend
Training: bottom / v3_dow_effect
Training: bottom / v4_price_event_dept


,group,feature_set,model,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,v4_price_event_dept,XGBoost,0.431857,0.735203,0.339818,1.270848,2.163519
1,bottom,v3_dow_effect,XGBoost,0.432677,0.736923,0.339818,1.273262,2.168579
2,bottom,v2_trend,XGBoost,0.433099,0.735058,0.339818,1.274502,2.163091
3,bottom,base_lag_rolling,XGBoost,0.433459,0.742508,0.339818,1.275561,2.185015
4,middle,v4_price_event_dept,XGBoost,0.963666,1.403388,1.148424,0.839120,1.222011
5,middle,v3_dow_effect,XGBoost,0.967354,1.409760,1.148424,0.842331,1.227560
6,middle,v2_trend,XGBoost,0.968667,1.410968,1.148424,0.843475,1.228612
7,middle,base_lag_rolling,XGBoost,0.968735,1.412191,1.148424,0.843534,1.229677
8,top,v4_price_event_dept,XGBoost,5.306233,8.506897,16.138667,0.328790,0.527113
9,top,v3_dow_effect,XGBoost,5.361279,8.578793,16.138667,0.332201,0.531568


In [25]:
# ============================================================
# 25. Best feature set and feature importance
# ============================================================

best_summary = feature_compare_summary.sort_values(
    ["group", "mae_ratio"]
).groupby("group").head(1).reset_index(drop=True)

display(best_summary)

best_results = []

for _, row in best_summary.iterrows():
    group_name = row["group"]
    feature_set_name = row["feature_set"]

    matched = [
        r for r in all_results
        if r["group"] == group_name and r["feature_set"] == feature_set_name
    ][0]

    best_results.append(matched)

importance_best_all = pd.concat(
    [r["importance"] for r in best_results],
    ignore_index=True
)

for group_name in ["top", "middle", "bottom"]:
    print("\n====", group_name, "====")
    display(
        importance_best_all[
            importance_best_all["group"] == group_name
        ].head(25)
    )

,group,feature_set,model,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,v4_price_event_dept,XGBoost,0.431857,0.735203,0.339818,1.270848,2.163519
1,middle,v4_price_event_dept,XGBoost,0.963666,1.403388,1.148424,0.839120,1.222011
2,top,v4_price_event_dept,XGBoost,5.306233,8.506897,16.138667,0.328790,0.527113



==== top ====


,feature,importance,group,model
168,dow_ratio_x_rolling_mean_7,0.572722,top,XGBoost
169,rolling_mean_7,0.124302,top,XGBoost
170,price_x_rolling_mean_7,0.019116,top,XGBoost
171,lag_21,0.015481,top,XGBoost
172,weekend_x_rolling_mean_7,0.011920,top,XGBoost
173,rolling_mean_28,0.011316,top,XGBoost
174,lag_14,0.009542,top,XGBoost
175,lag_28,0.009389,top,XGBoost
176,sell_price,0.009341,top,XGBoost
177,has_event,0.008716,top,XGBoost



==== middle ====


,feature,importance,group,model
84,dow_ratio_x_rolling_mean_7,0.381479,middle,XGBoost
85,zero_rate_7,0.157879,middle,XGBoost
86,rolling_mean_14,0.118424,middle,XGBoost
87,rolling_mean_28,0.040599,middle,XGBoost
88,snap,0.016928,middle,XGBoost
89,quarter,0.013200,middle,XGBoost
90,weekend_x_rolling_mean_7,0.010876,middle,XGBoost
91,zero_rate_14,0.010177,middle,XGBoost
92,store_WI_2,0.009148,middle,XGBoost
93,rolling_mean_7_div_28,0.009051,middle,XGBoost



==== bottom ====


,feature,importance,group,model
0,rolling_mean_14,0.171816,bottom,XGBoost
1,rolling_mean_28,0.109357,bottom,XGBoost
2,dow_ratio_x_rolling_mean_7,0.086328,bottom,XGBoost
3,rolling_mean_7,0.033551,bottom,XGBoost
4,weekend_x_rolling_mean_7,0.019733,bottom,XGBoost
5,zero_rate_28,0.013465,bottom,XGBoost
6,event_type_1_Sporting,0.013057,bottom,XGBoost
7,month,0.012748,bottom,XGBoost
8,zero_rate_7,0.012382,bottom,XGBoost
9,store_TX_2,0.012331,bottom,XGBoost


In [26]:
# ============================================================
# 26. Walk-forward validation
# ============================================================

walk_forward_splits = [
    pd.to_datetime("2015-10-01"),
    pd.to_datetime("2015-12-01"),
    pd.to_datetime("2016-02-01"),
    pd.to_datetime("2016-03-01")
]

def walk_forward_evaluate(df, group_name, features, feature_set_name, split_dates):
    rows = []

    for split in split_dates:
        print(f"Walk-forward: {group_name} / {split.date()}")

        result = train_ml_model(
            df=df,
            group_name=group_name,
            features=features,
            model_name="XGBoost",
            split_date=split
        )

        if result is None:
            continue

        rows.append({
            "group": group_name,
            "feature_set": feature_set_name,
            "split_date": split,
            "mae": result["mae"],
            "rmse": result["rmse"],
            "test_sales_mean": result["test_sales_mean"],
            "mae_ratio": result["mae_ratio"],
            "rmse_ratio": result["rmse_ratio"]
        })

    return pd.DataFrame(rows)


walk_forward_results = pd.concat([
    walk_forward_evaluate(
        model_df_ml,
        "top",
        features_v4,
        "v4_price_event_dept",
        walk_forward_splits
    ),
    walk_forward_evaluate(
        model_df_ml,
        "middle",
        features_v4,
        "v4_price_event_dept",
        walk_forward_splits
    ),
    walk_forward_evaluate(
        model_df_ml,
        "bottom",
        features_v4,
        "v4_price_event_dept",
        walk_forward_splits
    )
], ignore_index=True)

display(walk_forward_results)

walk_forward_summary = walk_forward_results.groupby("group").agg(
    mae_mean=("mae", "mean"),
    mae_std=("mae", "std"),
    rmse_mean=("rmse", "mean"),
    rmse_std=("rmse", "std"),
    mae_ratio_mean=("mae_ratio", "mean"),
    mae_ratio_std=("mae_ratio", "std"),
    rmse_ratio_mean=("rmse_ratio", "mean"),
    rmse_ratio_std=("rmse_ratio", "std")
).reset_index()

display(walk_forward_summary)

Walk-forward: top / 2015-10-01
Walk-forward: top / 2015-12-01
Walk-forward: top / 2016-02-01
Walk-forward: top / 2016-03-01
Walk-forward: middle / 2015-10-01
Walk-forward: middle / 2015-12-01
Walk-forward: middle / 2016-02-01
Walk-forward: middle / 2016-03-01
Walk-forward: bottom / 2015-10-01
Walk-forward: bottom / 2015-12-01
Walk-forward: bottom / 2016-02-01
Walk-forward: bottom / 2016-03-01


,group,feature_set,split_date,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,top,v4_price_event_dept,2015-10-01,5.264091,9.096062,15.170177,0.347003,0.599602
1,top,v4_price_event_dept,2015-12-01,5.163664,8.815845,14.966598,0.345013,0.589035
2,top,v4_price_event_dept,2016-02-01,5.363871,8.738124,16.236548,0.330358,0.538176
3,top,v4_price_event_dept,2016-03-01,5.306233,8.506897,16.138667,0.328790,0.527113
4,middle,v4_price_event_dept,2015-10-01,0.908262,1.341168,1.041948,0.871696,1.287173
5,middle,v4_price_event_dept,2015-12-01,0.911881,1.356095,1.057466,0.862327,1.282401
6,middle,v4_price_event_dept,2016-02-01,0.944142,1.386161,1.127024,0.837730,1.229931
7,middle,v4_price_event_dept,2016-03-01,0.963666,1.403388,1.148424,0.839120,1.222011
8,bottom,v4_price_event_dept,2015-10-01,0.395236,0.656417,0.295346,1.338213,2.222536
9,bottom,v4_price_event_dept,2015-12-01,0.403327,0.675015,0.307511,1.311584,2.195089


,group,mae_mean,mae_std,rmse_mean,rmse_std,mae_ratio_mean,mae_ratio_std,rmse_ratio_mean,rmse_ratio_std
0,bottom,0.412670,0.016502,0.694143,0.035235,1.302611,0.028989,2.189995,0.025235
1,middle,0.931988,0.026573,1.371703,0.028219,0.852718,0.016951,1.255379,0.034166
2,top,5.274465,0.084434,8.789232,0.243021,0.337791,0.009544,0.563481,0.036151


In [117]:
# ============================================================
# セル20：特徴量リスト作成 修正版
# 文字列カラム dept_id / store_id / state_id / event_type_1 / event_type_2 を除外
# ============================================================

date_features = [
    "wday",
    "month",
    "year",
    "day",
    "dayofweek",
    "weekofyear",
    "quarter",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "week_of_month",
    "is_week_1",
    "is_week_2",
    "is_week_3",
    "is_week_4",
    "is_week_5"
]

lag_features = [
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_28"
]

rolling_features = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_28",
    "zero_rate_7",
    "zero_rate_14",
    "zero_rate_28"
]

trend_features = [
    "rolling_mean_7_minus_28",
    "rolling_mean_7_div_28",
    "lag_7_minus_14",
    "lag_7_minus_21",
    "lag_7_minus_28"
]

dow_effect_features = [
    "group_dow_sales_mean",
    "group_dow_sales_ratio",
    "dow_ratio_x_rolling_mean_7",
    "weekend_x_rolling_mean_7",
    "week_of_month_x_rolling_mean_7"
]

price_features = [
    "sell_price",
    "price_lag_1",
    "price_diff_1",
    "price_pct_change_1",
    "is_price_down",
    "is_price_up",
    "price_x_rolling_mean_7",
    "price_down_x_rolling_mean_7"
]

event_snap_features = [
    "has_event_1",
    "has_event_2",
    "has_event",
    "snap",
    "snap_x_rolling_mean_7",
    "event_x_rolling_mean_7"
]

# 元の文字列カラムは除外する
exclude_categorical_originals = {
    "dept_id",
    "store_id",
    "state_id",
    "event_type_1",
    "event_type_2",
    "event_name_1",
    "event_name_2",
    "item_id",
    "cat_id",
    "id",
    "d"
}

onehot_features = [
    col for col in model_df.columns
    if (
        col not in exclude_categorical_originals
        and (
            col.startswith("dept_")
            or col.startswith("store_")
            or col.startswith("state_")
            or col.startswith("dow_")
            or col.startswith("event_type_1_")
            or col.startswith("event_type_2_")
        )
    )
]

base_features = (
    date_features +
    lag_features +
    rolling_features
)

features_v2 = (
    base_features +
    trend_features
)

features_v3 = (
    features_v2 +
    dow_effect_features
)

features_v4 = (
    features_v3 +
    price_features +
    event_snap_features +
    onehot_features
)

base_features = [col for col in list(dict.fromkeys(base_features)) if col in model_df.columns]
features_v2 = [col for col in list(dict.fromkeys(features_v2)) if col in model_df.columns]
features_v3 = [col for col in list(dict.fromkeys(features_v3)) if col in model_df.columns]
features_v4 = [col for col in list(dict.fromkeys(features_v4)) if col in model_df.columns]

print("base:", len(base_features))
print("v2:", len(features_v2))
print("v3:", len(features_v3))
print("v4:", len(features_v4))

# 念のため、object/stringが混ざってないか確認
bad_cols = [
    col for col in features_v4
    if model_df[col].dtype == "object" or str(model_df[col].dtype) == "string"
]

print("bad object columns in features_v4:", bad_cols)

base: 29
v2: 34
v3: 39
v4: 84
bad object columns in features_v4: []


In [27]:
# ============================================================
# 27. Walk-forward plots
# ============================================================

for group_name in ["top", "middle", "bottom"]:
    plot_df = walk_forward_results[
        walk_forward_results["group"] == group_name
    ].copy()

    plt.figure(figsize=(8, 4))
    plt.plot(plot_df["split_date"], plot_df["mae_ratio"], marker="o")
    plt.title(f"Walk-forward MAE Ratio: {group_name}")
    plt.xlabel("Split Date")
    plt.ylabel("MAE Ratio")
    plt.grid(True)
    plt.tight_layout()

    image_path = f"images/walk_forward_mae_ratio_{group_name}.png"
    plt.savefig(image_path)
    plt.close()

    print("saved:", image_path)

saved: images/walk_forward_mae_ratio_top.png
saved: images/walk_forward_mae_ratio_middle.png
saved: images/walk_forward_mae_ratio_bottom.png


In [28]:
# ============================================================
# 28. Model variants: clip / log1p / poisson
# ============================================================

def train_xgb_with_clip(df, group_name, features, split_date):
    group_df = df[df["sales_group"] == group_name].copy()

    train_df = group_df[group_df["date"] < split_date].copy()
    test_df = group_df[group_df["date"] >= split_date].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        print(f"{group_name}: train or test is empty")
        return None

    X_train = train_df[features]
    y_train = train_df["sales"]

    X_test = test_df[features]
    y_test = test_df["sales"]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds_raw = model.predict(X_test)
    preds_clip = np.clip(preds_raw, 0, None)

    mae_raw = mean_absolute_error(y_test, preds_raw)
    rmse_raw = np.sqrt(mean_squared_error(y_test, preds_raw))

    mae_clip = mean_absolute_error(y_test, preds_clip)
    rmse_clip = np.sqrt(mean_squared_error(y_test, preds_clip))

    sales_mean = y_test.mean()

    return {
        "group": group_name,
        "model": "XGBoost_clip",
        "mae_raw": mae_raw,
        "rmse_raw": rmse_raw,
        "mae_clip": mae_clip,
        "rmse_clip": rmse_clip,
        "test_sales_mean": sales_mean,
        "mae_ratio_raw": mae_raw / sales_mean if sales_mean != 0 else np.nan,
        "mae_ratio_clip": mae_clip / sales_mean if sales_mean != 0 else np.nan,
        "rmse_ratio_raw": rmse_raw / sales_mean if sales_mean != 0 else np.nan,
        "rmse_ratio_clip": rmse_clip / sales_mean if sales_mean != 0 else np.nan,
        "negative_pred_rate": (preds_raw < 0).mean(),
        "trained_model": model,
        "test_df": test_df,
        "preds_raw": preds_raw,
        "preds_clip": preds_clip
    }


def train_xgb_log1p(df, group_name, features, split_date):
    group_df = df[df["sales_group"] == group_name].copy()

    train_df = group_df[group_df["date"] < split_date].copy()
    test_df = group_df[group_df["date"] >= split_date].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        print(f"{group_name}: train or test is empty")
        return None

    X_train = train_df[features]
    y_train = train_df["sales"]

    X_test = test_df[features]
    y_test = test_df["sales"]

    y_train_log = np.log1p(y_train)

    model = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )

    model.fit(X_train, y_train_log)

    preds_log = model.predict(X_test)
    preds = np.expm1(preds_log)
    preds = np.clip(preds, 0, None)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    sales_mean = y_test.mean()

    importance_df = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_,
        "group": group_name,
        "model": "XGBoost_log1p"
    }).sort_values(
        "importance",
        ascending=False
    ).reset_index(drop=True)

    return {
        "group": group_name,
        "model": "XGBoost_log1p",
        "mae": mae,
        "rmse": rmse,
        "test_sales_mean": sales_mean,
        "mae_ratio": mae / sales_mean if sales_mean != 0 else np.nan,
        "rmse_ratio": rmse / sales_mean if sales_mean != 0 else np.nan,
        "trained_model": model,
        "importance": importance_df,
        "test_df": test_df,
        "preds": preds
    }


def train_xgb_poisson(df, group_name, features, split_date):
    group_df = df[df["sales_group"] == group_name].copy()

    train_df = group_df[group_df["date"] < split_date].copy()
    test_df = group_df[group_df["date"] >= split_date].copy()

    if len(train_df) == 0 or len(test_df) == 0:
        print(f"{group_name}: train or test is empty")
        return None

    X_train = train_df[features]
    y_train = train_df["sales"]

    X_test = test_df[features]
    y_test = test_df["sales"]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective="count:poisson",
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    preds = np.clip(preds, 0, None)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    sales_mean = y_test.mean()

    importance_df = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_,
        "group": group_name,
        "model": "XGBoost_poisson"
    }).sort_values(
        "importance",
        ascending=False
    ).reset_index(drop=True)

    return {
        "group": group_name,
        "model": "XGBoost_poisson",
        "mae": mae,
        "rmse": rmse,
        "test_sales_mean": sales_mean,
        "mae_ratio": mae / sales_mean if sales_mean != 0 else np.nan,
        "rmse_ratio": rmse / sales_mean if sales_mean != 0 else np.nan,
        "trained_model": model,
        "importance": importance_df,
        "test_df": test_df,
        "preds": preds
    }

In [29]:
# ============================================================
# 29. Evaluate model variants
# ============================================================

clip_results = []
log1p_results = []
poisson_results = []

for group_name in ["top", "middle", "bottom"]:
    print("clip:", group_name)
    clip_results.append(
        train_xgb_with_clip(
            model_df_ml,
            group_name,
            features_v4,
            split_date
        )
    )

    print("log1p:", group_name)
    log1p_results.append(
        train_xgb_log1p(
            model_df_ml,
            group_name,
            features_v4,
            split_date
        )
    )

    print("poisson:", group_name)
    poisson_results.append(
        train_xgb_poisson(
            model_df_ml,
            group_name,
            features_v4,
            split_date
        )
    )

clip_summary = pd.DataFrame([
    {
        "group": r["group"],
        "mae_raw": r["mae_raw"],
        "mae_clip": r["mae_clip"],
        "mae_improvement": r["mae_raw"] - r["mae_clip"],
        "mae_improvement_rate": (r["mae_raw"] - r["mae_clip"]) / r["mae_raw"],
        "mae_ratio_raw": r["mae_ratio_raw"],
        "mae_ratio_clip": r["mae_ratio_clip"],
        "negative_pred_rate": r["negative_pred_rate"]
    }
    for r in clip_results
    if r is not None
])

log1p_summary = pd.DataFrame([
    {
        "group": r["group"],
        "model": r["model"],
        "mae": r["mae"],
        "rmse": r["rmse"],
        "test_sales_mean": r["test_sales_mean"],
        "mae_ratio": r["mae_ratio"],
        "rmse_ratio": r["rmse_ratio"]
    }
    for r in log1p_results
    if r is not None
])

poisson_summary = pd.DataFrame([
    {
        "group": r["group"],
        "model": r["model"],
        "mae": r["mae"],
        "rmse": r["rmse"],
        "test_sales_mean": r["test_sales_mean"],
        "mae_ratio": r["mae_ratio"],
        "rmse_ratio": r["rmse_ratio"]
    }
    for r in poisson_results
    if r is not None
])

display(clip_summary)
display(log1p_summary)
display(poisson_summary)

clip: top
log1p: top
poisson: top
clip: middle
log1p: middle
poisson: middle
clip: bottom
log1p: bottom
poisson: bottom


,group,mae_raw,mae_clip,mae_improvement,mae_improvement_rate,mae_ratio_raw,mae_ratio_clip,negative_pred_rate
0,top,5.306233,5.305883,0.00035,0.000066,0.328790,0.328768,0.002303
1,middle,0.963666,0.963626,0.00004,0.000042,0.839120,0.839085,0.004000
2,bottom,0.431857,0.431857,0.00000,0.000000,1.270848,1.270848,0.000000


,group,model,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,top,XGBoost_log1p,5.192724,8.908637,16.138667,0.321757,0.552006
1,middle,XGBoost_log1p,0.936399,1.447280,1.148424,0.815377,1.260231
2,bottom,XGBoost_log1p,0.398241,0.759403,0.339818,1.171923,2.234733


,group,model,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,top,XGBoost_poisson,5.309273,8.494277,16.138667,0.328978,0.526331
1,middle,XGBoost_poisson,0.963529,1.402135,1.148424,0.839001,1.220921
2,bottom,XGBoost_poisson,0.430827,0.736119,0.339818,1.267817,2.166215


In [30]:
# ============================================================
# 30. Final model comparison
# ============================================================

xgb_v4_summary = best_summary.copy()
xgb_v4_summary["model_variant"] = "XGBoost_squarederror_v4"

xgb_v4_summary_for_compare = xgb_v4_summary[
    [
        "group",
        "model_variant",
        "mae",
        "rmse",
        "test_sales_mean",
        "mae_ratio",
        "rmse_ratio"
    ]
].copy()

log1p_for_compare = log1p_summary.copy()
log1p_for_compare["model_variant"] = "XGBoost_log1p"

log1p_for_compare = log1p_for_compare[
    [
        "group",
        "model_variant",
        "mae",
        "rmse",
        "test_sales_mean",
        "mae_ratio",
        "rmse_ratio"
    ]
].copy()

poisson_for_compare = poisson_summary.copy()
poisson_for_compare["model_variant"] = "XGBoost_poisson"

poisson_for_compare = poisson_for_compare[
    [
        "group",
        "model_variant",
        "mae",
        "rmse",
        "test_sales_mean",
        "mae_ratio",
        "rmse_ratio"
    ]
].copy()

final_model_compare = pd.concat(
    [
        xgb_v4_summary_for_compare,
        log1p_for_compare,
        poisson_for_compare
    ],
    ignore_index=True
)

final_model_compare = final_model_compare.sort_values(
    ["group", "mae_ratio"]
).reset_index(drop=True)

display(final_model_compare)

final_best_by_group = final_model_compare.sort_values(
    ["group", "mae_ratio"]
).groupby("group").head(1).reset_index(drop=True)

display(final_best_by_group)

,group,model_variant,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,XGBoost_log1p,0.398241,0.759403,0.339818,1.171923,2.234733
1,bottom,XGBoost_poisson,0.430827,0.736119,0.339818,1.267817,2.166215
2,bottom,XGBoost_squarederror_v4,0.431857,0.735203,0.339818,1.270848,2.163519
3,middle,XGBoost_log1p,0.936399,1.447280,1.148424,0.815377,1.260231
4,middle,XGBoost_poisson,0.963529,1.402135,1.148424,0.839001,1.220921
5,middle,XGBoost_squarederror_v4,0.963666,1.403388,1.148424,0.839120,1.222011
6,top,XGBoost_log1p,5.192724,8.908637,16.138667,0.321757,0.552006
7,top,XGBoost_squarederror_v4,5.306233,8.506897,16.138667,0.328790,0.527113
8,top,XGBoost_poisson,5.309273,8.494277,16.138667,0.328978,0.526331


,group,model_variant,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,XGBoost_log1p,0.398241,0.759403,0.339818,1.171923,2.234733
1,middle,XGBoost_log1p,0.936399,1.447280,1.148424,0.815377,1.260231
2,top,XGBoost_log1p,5.192724,8.908637,16.138667,0.321757,0.552006


In [31]:
# ============================================================
# 31. Save results and models
# ============================================================

Path("results").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)
Path("images").mkdir(exist_ok=True)

baseline_results.to_csv(
    "results/m5_baseline_results.csv",
    index=False,
    encoding="utf-8-sig"
)

feature_compare_summary.to_csv(
    "results/m5_feature_compare_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

best_summary.to_csv(
    "results/m5_best_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

importance_best_all.to_csv(
    "results/m5_best_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

dept_sales.to_csv(
    "results/m5_dept_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

week_of_month_sales.to_csv(
    "results/m5_week_of_month_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

event_sales.to_csv(
    "results/m5_event_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

snap_sales.to_csv(
    "results/m5_snap_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

price_down_sales.to_csv(
    "results/m5_price_down_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

walk_forward_results.to_csv(
    "results/m5_walk_forward_results.csv",
    index=False,
    encoding="utf-8-sig"
)

walk_forward_summary.to_csv(
    "results/m5_walk_forward_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

clip_summary.to_csv(
    "results/m5_clip_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

log1p_summary.to_csv(
    "results/m5_log1p_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

poisson_summary.to_csv(
    "results/m5_poisson_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

final_model_compare.to_csv(
    "results/m5_final_model_compare.csv",
    index=False,
    encoding="utf-8-sig"
)

final_best_by_group.to_csv(
    "results/m5_final_best_by_group.csv",
    index=False,
    encoding="utf-8-sig"
)

# log1p best models are saved for later use
for result in log1p_results:
    if result is None:
        continue

    group_name = result["group"]
    model = result["trained_model"]

    model_path = f"models/xgboost_log1p_{group_name}.joblib"
    joblib.dump(model, model_path)

    print("saved model:", model_path)

print("saved all results")

saved model: models/xgboost_log1p_top.joblib
saved model: models/xgboost_log1p_middle.joblib
saved model: models/xgboost_log1p_bottom.joblib
saved all results


In [32]:
# ============================================================
# 32. Final summary output
# ============================================================

print("==== Feature compare summary ====")
display(feature_compare_summary)

print("==== Best feature set by group ====")
display(best_summary)

print("==== Final model comparison ====")
display(final_model_compare)

print("==== Final best model by group ====")
display(final_best_by_group)

print("==== Saved files in results/ ====")
for p in sorted(Path("results").glob("*")):
    print(p)

print("==== Saved files in models/ ====")
for p in sorted(Path("models").glob("*")):
    print(p)

print("==== Saved files in images/ ====")
for p in sorted(Path("images").glob("*")):
    print(p)

==== Feature compare summary ====


,group,feature_set,model,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,v4_price_event_dept,XGBoost,0.431857,0.735203,0.339818,1.270848,2.163519
1,bottom,v3_dow_effect,XGBoost,0.432677,0.736923,0.339818,1.273262,2.168579
2,bottom,v2_trend,XGBoost,0.433099,0.735058,0.339818,1.274502,2.163091
3,bottom,base_lag_rolling,XGBoost,0.433459,0.742508,0.339818,1.275561,2.185015
4,middle,v4_price_event_dept,XGBoost,0.963666,1.403388,1.148424,0.839120,1.222011
5,middle,v3_dow_effect,XGBoost,0.967354,1.409760,1.148424,0.842331,1.227560
6,middle,v2_trend,XGBoost,0.968667,1.410968,1.148424,0.843475,1.228612
7,middle,base_lag_rolling,XGBoost,0.968735,1.412191,1.148424,0.843534,1.229677
8,top,v4_price_event_dept,XGBoost,5.306233,8.506897,16.138667,0.328790,0.527113
9,top,v3_dow_effect,XGBoost,5.361279,8.578793,16.138667,0.332201,0.531568


==== Best feature set by group ====


,group,feature_set,model,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,v4_price_event_dept,XGBoost,0.431857,0.735203,0.339818,1.270848,2.163519
1,middle,v4_price_event_dept,XGBoost,0.963666,1.403388,1.148424,0.839120,1.222011
2,top,v4_price_event_dept,XGBoost,5.306233,8.506897,16.138667,0.328790,0.527113


==== Final model comparison ====


,group,model_variant,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,XGBoost_log1p,0.398241,0.759403,0.339818,1.171923,2.234733
1,bottom,XGBoost_poisson,0.430827,0.736119,0.339818,1.267817,2.166215
2,bottom,XGBoost_squarederror_v4,0.431857,0.735203,0.339818,1.270848,2.163519
3,middle,XGBoost_log1p,0.936399,1.447280,1.148424,0.815377,1.260231
4,middle,XGBoost_poisson,0.963529,1.402135,1.148424,0.839001,1.220921
5,middle,XGBoost_squarederror_v4,0.963666,1.403388,1.148424,0.839120,1.222011
6,top,XGBoost_log1p,5.192724,8.908637,16.138667,0.321757,0.552006
7,top,XGBoost_squarederror_v4,5.306233,8.506897,16.138667,0.328790,0.527113
8,top,XGBoost_poisson,5.309273,8.494277,16.138667,0.328978,0.526331


==== Final best model by group ====


,group,model_variant,mae,rmse,test_sales_mean,mae_ratio,rmse_ratio
0,bottom,XGBoost_log1p,0.398241,0.759403,0.339818,1.171923,2.234733
1,middle,XGBoost_log1p,0.936399,1.447280,1.148424,0.815377,1.260231
2,top,XGBoost_log1p,5.192724,8.908637,16.138667,0.321757,0.552006


==== Saved files in results/ ====
results\m5_baseline_results.csv
results\m5_best_feature_importance.csv
results\m5_best_summary.csv
results\m5_clip_summary.csv
results\m5_dept_sales.csv
results\m5_event_sales.csv
results\m5_feature_compare_summary.csv
results\m5_final_best_by_group.csv
results\m5_final_model_compare.csv
results\m5_log1p_summary.csv
results\m5_poisson_summary.csv
results\m5_price_down_sales.csv
results\m5_snap_sales.csv
results\m5_walk_forward_results.csv
results\m5_walk_forward_summary.csv
results\m5_week_of_month_sales.csv
==== Saved files in models/ ====
models\xgboost_log1p_bottom.joblib
models\xgboost_log1p_middle.joblib
models\xgboost_log1p_top.joblib
==== Saved files in images/ ====
images\walk_forward_mae_ratio_bottom.png
images\walk_forward_mae_ratio_middle.png
images\walk_forward_mae_ratio_top.png
